# K3 · Tối ưu siêu tham số bằng Optuna trên Qwen QLoRA (T11)
8 trial × (30% dữ liệu train, tối đa 120 bước), mục tiêu = val loss, MedianPruner cắt trial kém sớm. ~2–3 giờ trên T4. Cấu hình tốt nhất lưu `results/optuna_best.json` và được K4 dùng cho cả 4 ô ma trận.

In [ ]:
%%capture
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.17.0" "bitsandbytes==0.47.0" "accelerate>=1.4" "datasets>=3.0" rank-bm25 optuna

In [ ]:
import sys, glob, json, os, time
sys.path.insert(0, glob.glob('/kaggle/input/**/code/src', recursive=True)[0])
from kimochi.kaggle_env import setup, gpu_info
ROOT = setup()
import pandas as pd
from pathlib import Path
RES = ROOT / 'results'
print(gpu_info())

In [ ]:
N_TRIALS, FRAC, MAX_STEPS = 8, 0.3, 120
import optuna
from transformers import TrainerCallback
from kimochi import llm

class OptunaPrune(TrainerCallback):
    def __init__(self, trial): self.trial = trial
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        self.trial.report(metrics['eval_loss'], state.global_step)
        if self.trial.should_prune():
            raise optuna.TrialPruned()

def objective(trial):
    r = trial.suggest_categorical('r', [8, 16, 32])
    lr = trial.suggest_float('lr', 5e-5, 3e-4, log=True)
    out = f'/kaggle/working/optuna/t{trial.number}'
    try:
        res = llm.train(ROOT, 'qwen', 'nf4', out, lr=lr, r=r, max_steps=MAX_STEPS, data_frac=FRAC, eval_steps=30,
                        extra_callbacks=[OptunaPrune(trial)])
    finally:
        import gc, torch; gc.collect(); torch.cuda.empty_cache()
    trial.set_user_attr('train_minutes', res['train_minutes']); trial.set_user_attr('peak_vram_gb', res['peak_vram_gb'])
    return res['eval_loss']

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=30))
study.optimize(objective, n_trials=N_TRIALS)

In [ ]:
df = study.trials_dataframe(); df.to_csv(RES / 'optuna_trials.csv', index=False)
(RES / 'optuna_best.json').write_text(json.dumps({'best_params': study.best_params, 'best_val_loss': study.best_value}, indent=1))
display(df[['number', 'value', 'params_r', 'params_lr', 'state']]); study.best_params

In [ ]:
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
ax = plot_optimization_history(study); plt.savefig(RES / 'optuna_history.png', dpi=200, bbox_inches='tight')